# Creating annotated 1-kb windows of the H37Rv genome

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

%matplotlib inline

In [2]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf
import bioframe.vis

#### Pandas Viewing Settings

In [3]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

### Import custom functions

In [4]:
%load_ext autoreload
%autoreload 2

from gcutils.general import label_DF_ByOvrLapGenes

# 1) Parse processed H37rv genome annotations

In [5]:
RepoRef_Dir = "../../References"
AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"

H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

## H37Rv Gene Annotations TSV
H37Rv_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")

# Add Middle and length to the annotations TSV
H37Rv_Genes_DF["Middle"] = (H37Rv_Genes_DF["Start"] + H37Rv_Genes_DF["End"]) / 2
H37Rv_Genes_DF["Length"]  = H37Rv_Genes_DF["End"] - H37Rv_Genes_DF["Start"]

### Output Rv annotations w/ extra 2 columns

In [6]:
RepoRef_Dir = "../../References"

H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.WiCtrAndLen.tsv"

H37Rv_Genes_DF.to_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t", index = False)

# 2) Annotate 1-kb windows of H37Rv by overlapping genes

### 2.A) Parse 1-kb window DF of Rv genome

In [7]:
# Define path to 1000bp window BED file for H37rv
RepoRef_Dir = "../../References"
H37Rv_Windows_Dir = f"{RepoRef_Dir}/H37Rv_GenomeWindows"

H37rv_1000bp_Window_BED = f"{H37Rv_Windows_Dir}/H37Rv.1000bp.windows.bed"


In [8]:
print(H37rv_1000bp_Window_BED)

../../References/H37Rv_GenomeWindows/H37Rv.1000bp.windows.bed


### 2.B) Annotate 1-kb windows of H37Rv

In [9]:
Rv_1kb_Win_DF = pd.read_csv(H37rv_1000bp_Window_BED, sep = "\t", header=None)
Rv_1kb_Win_DF.columns = ["Chrom", "Start", "End"]
Rv_1kb_Win_DF.shape

(4412, 3)

In [10]:
Rv_1kb_Anno_DF = label_DF_ByOvrLapGenes(Rv_1kb_Win_DF, H37Rv_Genes_DF)

Rv_1kb_Anno_DF["Middle"] = (Rv_1kb_Anno_DF["End"] + Rv_1kb_Anno_DF["Start"]) / 2
Rv_1kb_Anno_DF.shape

(4412, 5)

In [11]:
Rv_1kb_Anno_DF.head(4)

,Chrom,Start,End,Overlap_Genes,Middle
0,NC_000962.3,0,1000,dnaA,500.0
1,NC_000962.3,1000,2000,dnaA,1500.0
2,NC_000962.3,2000,3000,dnaN,2500.0
3,NC_000962.3,3000,4000,"dnaN,recF",3500.0


### 2.C) Output annotated windows of the Mtb genome

In [12]:
RepoRef_Dir = "../../References"
H37Rv_Windows_Dir = f"{RepoRef_Dir}/H37Rv_GenomeWindows"
!mkdir $H37Rv_Windows_Dir

H37Rv_1kb_Win_Anno_TSV = f"{H37Rv_Windows_Dir}/H37Rv.1000bp.Windows.Anno.tsv"

Rv_1kb_Anno_DF.to_csv(H37Rv_1kb_Win_Anno_TSV, sep = "\t", index=False)

mkdir: cannot create directory ‘../../References/H37Rv_GenomeWindows’: File exists


In [13]:
!ls -lah $H37Rv_Windows_Dir

total 331K
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Mar 10 17:43 .
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Mar 10 17:38 ..
-rw-r--r-- 1 mm774 hpc_farhat 212K Mar 10 17:45 H37Rv.1000bp.Windows.Anno.tsv
-rw-r--r-- 1 mm774 hpc_farhat 119K Mar 10 17:43 H37Rv.1000bp.windows.bed


## Read in annotated windows of the H37Rv genome

In [14]:
RepoRef_Dir = "../../References"

H37Rv_Windows_Dir = f"{RepoRef_Dir}/H37Rv_GenomeWindows"

H37Rv_1kb_Win_Anno_TSV = f"{H37Rv_Windows_Dir}/H37Rv.1000bp.Windows.Anno.tsv"

Rv_1kb_Anno_DF = pd.read_csv(H37Rv_1kb_Win_Anno_TSV, sep = "\t")


In [15]:
!head -n 5 $H37Rv_1kb_Win_Anno_TSV

Chrom	Start	End	Overlap_Genes	Middle
NC_000962.3	0	1000	dnaA	500.0
NC_000962.3	1000	2000	dnaA	1500.0
NC_000962.3	2000	3000	dnaN	2500.0
NC_000962.3	3000	4000	dnaN,recF	3500.0
